# Week 14: wavelet denoising

**What we do.** We remove white Gaussian noise from a 1D signal and from a photograph by thresholding wavelet coefficients (Donoho–Johnstone). We estimate the noise level from the data, compare the VisuShrink and SURE thresholds and hard and soft thresholding, and compare wavelet denoising of an image with the Gaussian and median filters of week 3.

**What to expect.** Noise spreads evenly over all wavelet coefficients, while the signal is concentrated in a few large ones; thresholding the details removes most of the noise and keeps sharp features. SURE adapts the threshold to each level and usually beats the single universal threshold. For a photograph the simple VisuShrink rule over-smooths, so its PSNR is not better than that of a Gaussian filter; we will see why.

**How to read the code.** Functions come from `wavelets_lab.denoising` (test signals and denoising), `wavelets_lab.filters` (noise, classical filters, PSNR) and `wavelets_lab.images`. We also call PyWavelets once to look at the coefficients directly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pywt

from wavelets_lab import denoising, filters, images

## The test signal and its noisy version

**What we do.** We create the Doppler signal of length 2048 and add Gaussian noise with standard deviation 0.05.

**What to expect.** The Doppler signal oscillates very fast near `t = 0` and slowly near `t = 1`; the noise hides the fastest oscillations.

**How to read the code.** `denoising.doppler(n)` samples the formula; the noise is drawn with a fixed seed so the results are reproducible.

In [ ]:
clean = denoising.doppler(2048)
noisy = clean + 0.05 * np.random.default_rng(0).standard_normal(clean.size)
t = np.arange(clean.size) / clean.size
plt.figure(figsize=(10, 3.5))
plt.plot(t, noisy, alpha=0.6, label="noisy")
plt.plot(t, clean, "k", label="clean")
plt.xlabel("t")
plt.ylabel("value")
plt.legend()
plt.show()

## Noise in the wavelet domain

**What we do.** We compare the finest detail coefficients `d_1` of the clean and the noisy signal and estimate the noise level from them.

**What to expect.** The clean `d_1` is almost zero except at the very beginning (fast oscillations); the noisy `d_1` is essentially pure noise with standard deviation 0.05 (the transform is orthonormal, so white noise stays white with the same σ). The robust estimate `median(|d_1|) / 0.6745` recovers σ ≈ 0.05.

**How to read the code.** `pywt.wavedec(..., level=6)[-1]` is the finest detail band; `denoising.estimate_sigma` is the MAD estimator.

In [ ]:
detail_clean = pywt.wavedec(clean, "db4", mode="periodization", level=6)[-1]
detail_noisy = pywt.wavedec(noisy, "db4", mode="periodization", level=6)[-1]
position = np.arange(detail_clean.size) * 2 / clean.size
plt.figure(figsize=(10, 3.5))
plt.plot(position, detail_noisy, ".", markersize=3, label="noisy d_1")
plt.plot(position, detail_clean, label="clean d_1")
plt.xlabel("position t")
plt.ylabel("detail coefficient")
plt.legend()
plt.show()
denoising.estimate_sigma(detail_noisy)

## VisuShrink and SURE

**What we do.** We denoise with the universal threshold (VisuShrink) and with level-dependent SURE thresholds, both with soft thresholding.

**What to expect.** Both remove most of the noise. VisuShrink is very smooth but loses some of the fast oscillations near `t = 0` (its threshold is large); SURE keeps more detail and has a smaller mean squared error (MSE, shown in the titles).

**How to read the code.** `denoising.denoise_signal(noisy, "db4", 6, method=...)` performs the DWT, the thresholding of every detail level and the inverse DWT. The loop plots the clean signal in light grey under each result.

In [ ]:
results = {
    "noisy": noisy,
    "VisuShrink": denoising.denoise_signal(noisy, "db4", 6, method="visu"),
    "SURE": denoising.denoise_signal(noisy, "db4", 6, method="sure"),
}
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for ax, (title, signal) in zip(axes, results.items(), strict=True):
    ax.plot(t, clean, "k", alpha=0.3)
    ax.plot(t, signal)
    ax.set_xlabel("t")
    ax.set_ylabel("value")
    ax.set_title(f"{title}: MSE {np.mean((signal - clean) ** 2):.2e}")
plt.tight_layout()
plt.show()

## Hard versus soft thresholding

**What we do.** We compare the MSE of the four combinations of method and thresholding rule on the noisy Doppler and HeaviSine signals.

**What to expect.** The differences depend on the signal. Soft thresholding gives smoother results; hard thresholding keeps the size of large coefficients (sharper jumps) but can leave isolated noise spikes. Compare the numbers and explain them.

**How to read the code.** Two nested loops call `denoising.denoise_signal` with all combinations of `method` and `mode` and print the MSE.

In [ ]:
signals = {"Doppler": clean, "HeaviSine": denoising.heavisine(2048) / 4}
for name, reference in signals.items():
    observed = reference + 0.05 * np.random.default_rng(1).standard_normal(
        reference.size
    )
    for method in ("visu", "sure"):
        for mode in ("hard", "soft"):
            result = denoising.denoise_signal(
                observed, "db4", 6, method=method, mode=mode
            )
            error = np.mean((result - reference) ** 2)
            print(f"{name:9s} {method} {mode}: MSE {error:.2e}")

## Denoising an image

**What we do.** We add noise with σ = 20 to `camera` and compare the Gaussian filter, the median filter and wavelet denoising with the universal threshold (VisuShrink, db4, 3 levels), with soft and with hard thresholding.

**What to expect.** All methods increase the PSNR by 3–5 dB. The universal threshold `σ sqrt(2 log n)` is designed to remove the noise almost surely, so it also removes weak image detail (grass texture): with soft thresholding the result is smooth but its PSNR (about 25 dB) is below that of the 5×5 Gaussian filter (about 27 dB); hard thresholding keeps more detail and gains about 1 dB. Compare the details, too: the wavelet results remove the noise in flat regions (the sky) much more completely than the Gaussian and median filters, but they blur fine texture and show small blocky artefacts along edges. Better wavelet methods (level-dependent thresholds, cycle spinning) beat the classical filters clearly; trying them is a good extension.

**How to read the code.** `denoising.denoise_image(noisy, "db4", 3, mode)` estimates σ from the finest diagonal subband, thresholds all details with the universal threshold and reconstructs. The PSNR of each result is in its title; the second row zooms into a detail (rows 60–187, columns 200–327).

In [ ]:
camera = images.load_test_image("camera")
noisy_image = filters.add_gaussian_noise(camera, 20.0, rng=1)
image_results = {
    "noisy": noisy_image,
    "Gauss 5x5": filters.gaussian_filter(noisy_image, 5),
    "median 3x3": filters.median_filter(noisy_image, 3),
    "wavelet, soft": denoising.denoise_image(noisy_image, "db4", 3, "soft"),
    "wavelet, hard": denoising.denoise_image(noisy_image, "db4", 3, "hard"),
}
detail = (slice(60, 188), slice(200, 328))
fig, axes = plt.subplots(2, 5, figsize=(24, 10))
for column, (title, image) in enumerate(image_results.items()):
    images.show_image(
        image, f"{title}: {filters.psnr(camera, image):.1f} dB", ax=axes[0, column]
    )
    images.show_image(image[detail], f"{title} (detail)", ax=axes[1, column])
plt.tight_layout()
plt.show()

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** Four modules are involved: `wavelets_lab.denoising` (test signals and denoising), `wavelets_lab.compression` (the thresholding rules used inside the denoising functions), `wavelets_lab.filters` (noise, classical filters, PSNR) and `wavelets_lab.images` (loading, display).

**How to read the code.** Functions marked *used internally* are called by `denoise_signal` and `denoise_image`.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `pywt.wavedec(x, wavelet, mode, level)` | multilevel 1D DWT | signal → list `[a_J, d_J, …, d_1]` |
| `np.random.default_rng(seed).standard_normal(n)` | Gaussian noise | seed, length → array |
| `np.mean((a - b) ** 2)` | mean squared error | arrays → number |

### Module `wavelets_lab.denoising` (file `src/wavelets_lab/denoising.py`)

Wavelet denoising (Donoho–Johnstone): white noise spreads evenly over all wavelet coefficients, while a piecewise smooth signal is concentrated in a few large ones; so thresholding the details removes most of the noise and keeps the signal. Imports:
```python
import numpy as np
import pywt
from wavelets_lab.compression import hard_threshold, soft_threshold
```
and the constant `MAD_CONSTANT = 0.6744897501960817`, the median of `|Z|` for a standard normal `Z`.

#### `denoising.doppler(n=1024)`

**What it is.** The Donoho–Johnstone "Doppler" test signal `sqrt(t(1 - t)) sin(2.1π / (t + 0.05))` on `[0, 1)`: its frequency decreases from very high near `t = 0` to low near `t = 1`, so it needs different time resolutions in different places.

**Input.** `n` (`int`): number of samples.

**Output.** `float64` array of length `n`.

**How to write it yourself.**
1. `t = np.arange(n) / n`.
2. Return `np.sqrt(t * (1 - t)) * np.sin(2.1 * np.pi / (t + 0.05))`.

#### `denoising.heavisine(n=1024)`

**What it is.** The "HeaviSine" test signal `4 sin(4πt) - sign(t - 0.3) - sign(0.72 - t)` on `[0, 1)`: a smooth sinusoid with two jumps (at `t = 0.3` and `t = 0.72`).

**Input.** `n` (`int`): number of samples.

**Output.** `float64` array of length `n`.

**How to write it yourself.**
1. `t = np.arange(n) / n`.
2. Return `4 * np.sin(4 * np.pi * t) - np.sign(t - 0.3) - np.sign(0.72 - t)`.

#### `denoising.estimate_sigma(detail)` (used internally)

**What it is.** Robust estimate of the noise standard deviation from the finest detail coefficients: `σ ≈ median(|d|) / 0.6745` (median absolute deviation, MAD). The finest details of a smooth signal are almost pure noise, and the median ignores the few large coefficients caused by edges.

**Input.** `detail` (array-like): finest-level detail coefficients.

**Output.** `float`.

**How to write it yourself.** Return `float(np.median(np.abs(detail)) / 0.6744897501960817)`.

#### `denoising.universal_threshold(sigma, n)` (used internally)

**What it is.** The universal (VisuShrink) threshold `σ sqrt(2 log n)`: with high probability the largest of `n` pure-noise coefficients stays below it, so thresholding at this level removes essentially all noise (and tends to over-smooth slightly).

**Input.** `sigma` (`float`): noise standard deviation; `n` (`int`): number of samples.

**Output.** `float`.

**How to write it yourself.** Return `sigma * np.sqrt(2 * np.log(n))`.

#### `denoising.sure_threshold(detail, sigma)` (used internally)

**What it is.** The soft threshold minimizing Stein's Unbiased Risk Estimate (SureShrink), chosen separately for each level. For coefficients `y_i = d_i / σ` (noise variance 1) and threshold `t`,
`SURE(t) = n - 2 #{i : |y_i| <= t} + sum_i min(|y_i|, t)^2`
is an unbiased estimate of the mean squared error of soft thresholding. It only changes at `t = |y_i|`, so it is enough to check `t = 0` and `t = |y_i|`. The result is capped by the universal threshold.

**Input.** `detail` (array-like): detail coefficients of one level; `sigma` (`float`).

**Output.** `float` threshold (on the original scale, i.e. multiplied by `σ`); `0.0` for empty input or `sigma <= 0`.

**How to write it yourself.**
1. `values = np.abs(np.ravel(detail))`, `n = values.size`; return `0.0` if `n == 0` or `sigma <= 0`.
2. Sorted squared normalized values: `scaled = np.sort(values / sigma) ** 2`.
3. Candidates `t^2`: `candidates = [0, scaled...]`; cumulative sums `cumulative = [0, cumsum(scaled)...]`; `counts = np.arange(n + 1)` (how many values are `<=` the candidate).
4. For the `j`-th candidate, `sum min(|y_i|, t)^2 = cumulative[j] + (n - j) * candidates[j]`, so `risks = n - 2 * counts + cumulative + (n - counts) * candidates`.
5. `best = sqrt(candidates[argmin(risks)]) * sigma`; return `min(best, universal_threshold(sigma, n))`.

#### `denoising.denoise_signal(x, wavelet="db4", level=5, method="visu", mode="soft")`

**What it is.** Removes white Gaussian noise from a 1D signal: DWT, threshold the details of every level, inverse DWT. The approximation is not thresholded.

**Input.**
- `x` (array-like): noisy signal with length divisible by `2**level`.
- `wavelet` (`str`), `level` (`int`).
- `method` (`"visu"` or `"sure"`): one universal threshold for all levels, or a SURE threshold for each level.
- `mode` (`"hard"` or `"soft"`).

**Output.** `float64` denoised signal. Raises `ValueError` for an unknown method or mode.

**How to write it yourself.**
1. Validate `method`; pick `rule = {"hard": hard_threshold, "soft": soft_threshold}[mode]` (raise `ValueError` if missing).
2. `coefficients = pywt.wavedec(x, wavelet, mode="periodization", level=level)`.
3. Noise level from the finest details: `sigma = estimate_sigma(coefficients[-1])`.
4. Keep `coefficients[0]`; for each detail array: threshold `t = universal_threshold(sigma, len(x))` (VisuShrink) or `t = sure_threshold(detail, sigma)` (SURE), and append `rule(detail, t)`.
5. Return `pywt.waverec(cleaned, wavelet, mode="periodization")`.

#### `denoising.denoise_image(image, wavelet="db4", level=3, mode="soft")`

**What it is.** VisuShrink denoising of an image with the 2D DWT.

**Input.** `image` (array-like): noisy greyscale image with sides divisible by `2**level`; `wavelet` (`str`); `level` (`int`); `mode` (`"hard"` or `"soft"`).

**Output.** `float64` denoised image. Raises `ValueError` for an unknown mode.

**How to write it yourself.**
1. Pick the rule as in `denoise_signal`.
2. `coefficients = pywt.wavedec2(image, wavelet, mode="periodization", level=level)`.
3. `sigma = estimate_sigma(coefficients[-1][2])` (the finest diagonal subband `cD_1`, which contains the least image structure).
4. `t = universal_threshold(sigma, image.size)`.
5. Keep `coefficients[0]`, apply `rule(band, t)` to every band of every detail tuple, and return `pywt.waverec2(cleaned, wavelet, mode="periodization")`.

### Module `wavelets_lab.compression` (file `src/wavelets_lab/compression.py`)

Thresholding of wavelet coefficients and wavelet image compression. As in the lecture notes, the **compression ratio** is the fraction of wavelet coefficients set to zero among all coefficients (for an orthogonal transform there are as many coefficients as pixels). Imports:
```python
import numpy as np
from wavelets_lab.dwt import decompose_image, reconstruct_image
```
Internal helper `_details(coefficients)`: all detail coefficients of a 2D DWT concatenated into one 1D array (`np.concatenate([np.ravel(band) for bands in coefficients[1:] for band in bands])`).

#### `compression.hard_threshold(x, threshold)`

**What it is.** Hard thresholding: values with `|x| < threshold` become 0, the others are kept unchanged ("keep or kill"). Same as `pywt.threshold(x, t, "hard")`.

**Input.** `x` (array-like): coefficients; `threshold` (`float`, non-negative).

**Output.** `float64` array.

**How to write it yourself.** Return `np.where(np.abs(x) >= threshold, x, 0.0)`.

#### `compression.soft_threshold(x, threshold)`

**What it is.** Soft thresholding: small values become 0 and all others are shrunk towards zero by `threshold`, `sign(x) * max(|x| - threshold, 0)` ("shrink or kill"). The result is continuous in `x` (no jump at the threshold), but large coefficients lose `threshold` of their size. Same as `pywt.threshold(x, t, "soft")`.

**Input.** `x` (array-like), `threshold` (`float`, non-negative).

**Output.** `float64` array.

**How to write it yourself.** Return `np.sign(x) * np.maximum(np.abs(x) - threshold, 0.0)`.

### Module `wavelets_lab.filters` (file `src/wavelets_lab/filters.py`)

Noise, linear smoothing (convolution with a mask) and median filtering of images, plus quality measures. Images follow the convention of `wavelets_lab.images` (`float64` on `[0, 255]`). Imports:
```python
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
```
The module defines two constants from the lecture notes: `GAUSS_3X3 = [[1, 2, 1], [2, 4, 2], [1, 2, 1]] / 16` and `GAUSS_5X5 = [[2, 4, 5, 4, 2], [4, 9, 12, 9, 4], [5, 12, 15, 12, 5], [4, 9, 12, 9, 4], [2, 4, 5, 4, 2]] / 159`. Both are integer approximations of a Gaussian, divided by the sum of their entries so that they sum to 1 (a constant image stays unchanged).

#### `filters.add_gaussian_noise(image, sigma, rng=None)`

**What it is.** Adds white Gaussian noise: an independent normal random number with mean 0 and standard deviation `sigma` to every pixel. It models sensor and transmission noise.

**Input.**
- `image` (array-like): image.
- `sigma` (`float`): standard deviation of the noise, e.g. 20 (in intensity units).
- `rng` (`numpy.random.Generator`, `int` or `None`): random generator or seed; a fixed seed (e.g. `rng=0`) makes the example reproducible.

**Output.** `float64` noisy image of the same shape, **not** clipped (values may leave `[0, 255]`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. Create a generator: `generator = np.random.default_rng(rng)` (accepts a seed, an existing generator or `None`).
3. Draw standard normal numbers of the same shape: `generator.standard_normal(array.shape)`.
4. Return `array + sigma * noise`.

#### `filters.convolve2d(image, kernel)` (used internally)

**What it is.** 2D convolution of an image with a small mask: every output pixel is a weighted sum of the input pixels around it. At the border the image is extended by mirroring, so the output has the same shape as the input.

**Input.**
- `image` (array-like): greyscale image of shape `(R, C)`.
- `kernel` (array-like): 2D mask with **odd** side lengths `(k1, k2)`.

**Output.** `float64` array of shape `(R, C)`. Raises `ValueError` if the mask is not 2D with odd sides.

**How to write it yourself.**
1. Convert both arguments to `float64`; check `mask.ndim == 2` and that both sides are odd.
2. Pad the image by half the mask size on each side with mirroring: `np.pad(array, ((p0, p0), (p1, p1)), mode="symmetric")` with `p0, p1 = k1 // 2, k2 // 2`.
3. Take all windows of the mask's size: `windows = sliding_window_view(padded, mask.shape)` has shape `(R, C, k1, k2)`; `windows[i, j]` is the neighbourhood of pixel `(i, j)`.
4. Convolution uses the mask flipped in both directions, so compute `np.einsum("ijkl,kl->ij", windows, mask[::-1, ::-1])` (the sum over `k, l` of window times mask).
5. Return the result. (For symmetric masks such as the Gaussian ones, flipping changes nothing.)

#### `filters.gaussian_filter(image, size=3)`

**What it is.** Smooths (blurs) an image by convolution with the 3×3 or 5×5 Gaussian mask of the lecture notes. It weakens noise, but also softens edges.

**Input.**
- `image` (array-like): greyscale image.
- `size` (`3` or `5`, default 3): mask size; 5×5 smooths more.

**Output.** `float64` smoothed image of the same shape. Raises `ValueError` for other sizes.

**How to write it yourself.**
1. Put the two masks in a dictionary: `{3: GAUSS_3X3, 5: GAUSS_5X5}`.
2. If `size` is not a key, raise `ValueError("size must be 3 or 5")`.
3. Return `convolve2d(image, masks[size])`.

#### `filters.median_filter(image, size=3)`

**What it is.** Replaces every pixel by the median of the `size × size` neighbourhood around it. Unlike averaging, the median ignores a few outliers, so it removes impulse noise and keeps edges sharp.

**Input.**
- `image` (array-like): greyscale image.
- `size` (`int`, positive and odd, default 3).

**Output.** `float64` filtered image of the same shape. Raises `ValueError` for an even or non-positive size.

**How to write it yourself.**
1. Check that `size` is positive and odd.
2. Pad the image with `np.pad(array, size // 2, mode="symmetric")`.
3. Build all windows: `sliding_window_view(padded, (size, size))`, shape `(R, C, size, size)`.
4. Take the median over the last two axes: `np.median(windows, axis=(2, 3))`.

#### `filters.mse(reference, image)` (used internally)

**What it is.** Mean squared error between two images of the same shape: `mean((reference - image) ** 2)`.

**Input.** Two array-like images of the same shape.

**Output.** `float`.

**How to write it yourself.**
1. Convert both to `float64` and subtract.
2. Square, take `np.mean`, and convert to a Python `float`.

#### `filters.psnr(reference, image, peak=255.0)`

**What it is.** Peak signal-to-noise ratio in decibels, `10 * log10(peak**2 / MSE)`, the standard measure of image quality: higher is better; +6 dB roughly means half the error amplitude. Identical images give infinity.

**Input.**
- `reference` (array-like): the clean image.
- `image` (array-like): the image to evaluate, same shape.
- `peak` (`float`, default 255): largest possible intensity.

**Output.** `float` (dB), `inf` if the images are equal.

**How to write it yourself.**
1. `error = mse(reference, image)`.
2. If `error == 0`, return `float("inf")`.
3. Return `10 * np.log10(peak**2 / error)` as a `float`.

### Module `wavelets_lab.images` (file `src/wavelets_lab/images.py`)

Images as NumPy arrays and simple pixel operations. **Convention used everywhere in the course:** a greyscale image is a 2D array of type `float64` with values on the scale `[0, 255]` (0 = black, 255 = white); a colour image is a 3D array of shape `(rows, cols, 3)` on the same scale. Working in `float64` (instead of the `uint8` stored in image files) avoids overflow and rounding during computations.

The module starts with the imports
```python
from pathlib import Path
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
```
and the constant `MAX_INTENSITY = 255.0`.

#### `images.load_test_image(name="camera")`

**What it is.** Loads one of the licence-free test images shipped with scikit-image (`skimage.data`), so that no image file is needed.

**Input.**
- `name` (`str`, default `"camera"`): name of a function in `skimage.data`, e.g. `"camera"` (512×512 greyscale), `"astronaut"` (512×512 RGB), `"coins"`, `"checkerboard"`.

**Output.** `numpy.ndarray` of type `float64` on the scale `[0, 255]` (2D for greyscale, `(rows, cols, 3)` for colour). Raises `ValueError` if the name does not exist.

**How to write it yourself.**
1. Import the module of test images: `from skimage import data`.
2. Look the loader up by name without failing: `loader = getattr(data, name, None)`.
3. If `loader` is `None` or not callable (`callable(loader)` is `False`), raise `ValueError(f"skimage.data has no test image called {name!r}")`.
4. Call it: `raw = loader()` returns a `uint8` array.
5. Return `to_float(raw)` (see `images.to_float`), which converts to `float64` on `[0, 255]`.

#### `images.to_float(image)` (used internally)

**What it is.** Converts any image array to the course convention: `float64` on `[0, 255]`.

**Input.**
- `image` (array-like): `uint8`, `bool`, integer or floating-point image.

**Output.** `numpy.ndarray` of type `float64`.

**How to write it yourself.**
1. `array = np.asarray(image)`.
2. If `array.dtype == np.bool_`: return `array.astype(np.float64) * 255.0` (True becomes white).
3. If the dtype is floating (`np.issubdtype(array.dtype, np.floating)`), the array is not empty and all values lie in `[0, 1]` (`array.min() >= 0` and `array.max() <= 1`): the image uses the unit scale, so return `array.astype(np.float64) * 255.0`.
4. Otherwise (integers, or floats already on a large scale) return `array.astype(np.float64)`.

#### `images.show_image(image, title=None, ax=None, scaled=False, xlabel="column [pixel]", ylabel="row [pixel]")`

**What it is.** Displays a greyscale or colour image with Matplotlib, using a 256-level grey colour map and **labelled axes** (by default pixel coordinates: column on the horizontal axis, row on the vertical axis, with row 0 at the top).

**Input.**
- `image` (array-like): 2D greyscale or `(rows, cols, 3)` RGB image on `[0, 255]`.
- `title` (`str` or `None`): title of the plot.
- `ax` (`matplotlib.axes.Axes` or `None`): where to draw; when `None` a new figure is created. Passing `ax` lets you place several images in one figure (`fig, axes = plt.subplots(1, 3)`).
- `scaled` (`bool`, default `False`): `False` maps 0 to black and 255 to white (true intensities); `True` stretches the smallest value of the data to black and the largest to white, like Matlab's `imagesc`. Use `True` for arrays that are not intensities (spectra, gradients, phases).
- `xlabel`, `ylabel` (`str`): axis labels; change them when the axes mean something else (e.g. angle and offset for a sinogram).

**Output.** The `Axes` object that contains the image (useful for further changes such as `ax.set_xlim`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. If `ax is None`: `_, ax = plt.subplots()`.
3. If `array.ndim == 3` (colour): `ax.imshow(to_uint8(array))`; Matplotlib then uses the RGB values directly.
4. Otherwise (greyscale): choose limits `vmin, vmax = (None, None)` if `scaled` else `(0.0, 255.0)` and call `ax.imshow(array, cmap=plt.get_cmap("gray", 256), vmin=vmin, vmax=vmax)`. With `vmin=vmax=None` Matplotlib uses the data minimum and maximum.
5. If a title is given: `ax.set_title(title)`.
6. Label both axes: `ax.set_xlabel(xlabel)` and `ax.set_ylabel(ylabel)`.
7. Return `ax`.

#### `images.to_uint8(image)` (used internally)

**What it is.** The inverse direction: prepares an image for saving or for `imshow` of colour images, which expects bytes.

**Input.**
- `image` (array-like): image on the scale `[0, 255]`, possibly with values outside it.

**Output.** `numpy.ndarray` of type `uint8`.

**How to write it yourself.**
1. Convert to floats: `np.asarray(image, dtype=np.float64)`.
2. Round to the nearest integer with `np.rint`.
3. Clip to the valid range with `np.clip(..., 0, 255)`; without this, e.g. 260 would wrap around to 4.
4. Cast with `.astype(np.uint8)` and return.